# `protein_based_graph` — quantile × criterion sweep (display only)

Ranking molecules by binding-profile information before the message-passing
**quantile filter**, then measuring the approved pipeline GNN across coverage
quantiles, for **both regimes** (inductive / transductive) and **both molecule
node sources** (ChemBERTa / GIN). **This notebook only displays results.** All
numbers are produced by

    scripts/modeling/train/run_quantile_criteria_sweep.py

which runs the exact ensemble-pipeline GNN (`GnnSignedExtractor` → `_SignedSage`,
5-model bag, emit=prot) with the shared 7 criteria (`orbind.mol_selection`) and
writes the CSVs read below. Re-run that script (once per regime × molecule
source) to refresh the data; nothing here trains anything.

In [ ]:
import sys, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as _t

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from orbind.mol_selection import CRITERIA

# --- what run_quantile_criteria_sweep.py produced ---
QUANTILES = [50, 80, 85, 90, 95, 99]
N_MODELS  = 5      # bagging flag: 5 = bagged bag (pipeline default), 1 = single model (no bagging)
METR      = ['AUROC', 'AUPRC', 'MCC', 'F1']
CACHE     = ROOT / 'results/graph/full_full/v7/protein_based_graph'

# regime -> (regime_key used in the filename, seeds the sweep ran)
REGIMES = {
    'inductive':    ('inductive_molecule_v5', [42, 43, 44, 45, 46]),
    'transductive': ('transductive',          [1, 2, 3, 4, 5]),
}
# molecule node source -> npz stem in the filename (also the raw-mol boost half)
MOL_SOURCES = {
    'ChemBERTa': 'chemberta_77m_m2or',
    'GIN':       'gin_supervised_contextpred_all_m2or',
}

def csv_for(regime, source):
    key = REGIMES[regime][0]
    nm = '' if N_MODELS == 5 else f'__nm{N_MODELS}'   # 5-model bag = legacy name
    return CACHE / f'metrics_{key}__{MOL_SOURCES[source]}{nm}.csv'

# --- styling: 7 criteria (categorical) + boost reference line ---
COL = {'coverage': '#2a78d6', 'balance_bits': '#eb6834', 'entropy_bits': '#1baf7a',
       'disc_pairs': '#eda100', 'idf_coverage': '#e87ba4', 'composite': '#008300',
       'greedy_pair_cover': '#4a3aa7', 'boost_full': '#0b0b0b'}
LABEL = {'boost_full': 'boost (full)'}
ORDER = list(CRITERIA) + ['boost_full']

def sty(c):
    if c == 'boost_full':
        return dict(ls='-', lw=2.6, marker='s', zorder=6)
    return dict(ls='-', lw=2.0, marker='o', zorder=3)

def mean_ci(vals):
    v = np.asarray(vals, float); n = len(v)
    if n < 2:
        return (v.mean() if n else np.nan), 0.0
    return v.mean(), _t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)

def _load(regime, source):
    csv = csv_for(regime, source)
    if not csv.exists():
        print(f'missing: {csv.name}  -- run the sweep for ({regime}, {source}) first')
        return None
    df = pd.read_csv(csv)
    if 'n_models' not in df.columns:
        df['n_models'] = 5              # legacy files predate the n_models column
    df = df[df['n_models'] == N_MODELS]
    if df.empty:
        print(f'{csv.name}: no rows at n_models={N_MODELS} (set N_MODELS to 1 or 5)')
        return None
    return df

def value_fig(regime, source):
    '''2x2 metric-value vs coverage-quantile, mean over seeds with 95% CI.'''
    df = _load(regime, source)
    if df is None:
        return
    seeds = REGIMES[regime][1]
    _dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, metric in zip(axes.ravel(), METR):
        for c in ORDER:
            sub = df[df.criterion == c]
            if sub.empty:
                continue
            qs = sorted(sub['quantile'].unique())
            means, errs = zip(*[mean_ci(sub[sub['quantile'] == q][metric].values) for q in qs])
            xd = [q + _dodge[c] for q in qs]; st = sty(c)
            ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
            ax.errorbar(xd, means, yerr=errs, fmt='none', ecolor=COL[c],
                        elinewidth=1.1, capsize=2, alpha=0.3, zorder=st['zorder'] - 1)
        ax.set_title(metric); ax.set_xlabel('coverage quantile')
        ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(ORDER),
               frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f'bagged GNN {source}-nodes (n_models={N_MODELS}) - {regime} - '
                 f'mean of seeds {seeds} (95% CI) - value vs quantile', y=1.06)
    fig.tight_layout(); plt.show()

def rank_fig(regime, source):
    '''2x2 mean-rank (1 = best) of each criterion vs coverage-quantile, per (quantile, seed).'''
    df = _load(regime, source)
    if df is None:
        return
    seeds = REGIMES[regime][1]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, metric in zip(axes.ravel(), METR):
        r = df.copy()
        r['rk'] = r.groupby(['quantile', 'seed'])[metric].rank(ascending=False, method='average')
        for c in ORDER:
            sub = r[r.criterion == c]
            if sub.empty:
                continue
            qs = sorted(sub['quantile'].unique())
            means = [sub[sub['quantile'] == q]['rk'].mean() for q in qs]
            ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
        ax.set_title(metric); ax.set_xlabel('coverage quantile')
        ax.set_ylabel('mean rank (1 = best)'); ax.set_xticks(QUANTILES)
        ax.invert_yaxis()
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(ORDER),
               frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f'bagged GNN {source}-nodes (n_models={N_MODELS}) - {regime} - '
                 f'mean of seeds {seeds} - mean rank vs quantile', y=1.06)
    fig.tight_layout(); plt.show()

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('display setup ready; CACHE =', CACHE)

## Inductive (`inductive_molecule_v5`) — the pipeline `cls+mol` feature

### ChemBERTa nodes

In [ ]:
value_fig('inductive', 'ChemBERTa')
rank_fig('inductive', 'ChemBERTa')

### GIN nodes

In [ ]:
value_fig('inductive', 'GIN')
rank_fig('inductive', 'GIN')

## Transductive

### ChemBERTa nodes

In [ ]:
value_fig('transductive', 'ChemBERTa')
rank_fig('transductive', 'ChemBERTa')

### GIN nodes

In [ ]:
value_fig('transductive', 'GIN')
rank_fig('transductive', 'GIN')